In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd()

# Si estás ejecutando desde notebooks/, subimos a la raíz del repo.
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CWD actual:", Path.cwd())

In [ ]:
from pathlib import Path

folders = [
    "src",
    "src/data",
    "src/visualization",
    "data",
    "data/hf_cache",
    "data/splits",
    "outputs",
    "outputs/dataset_checks",
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

# Archivos __init__.py para poder importar módulos desde src/
for init_file in [
    "src/__init__.py",
    "src/data/__init__.py",
    "src/visualization/__init__.py",
]:
    Path(init_file).touch()

print("Estructura creada correctamente.")

In [ ]:
%%writefile src/data/utils.py
import json
from pathlib import Path
from datasets import load_dataset


def load_mimic_dataset(cache_dir=None):
    """
    Carga el dataset MIMIC-CXR desde Hugging Face.
    """
    return load_dataset(
        "itsanmolgupta/mimic-cxr-dataset",
        cache_dir=cache_dir
    )


def is_empty_text(x):
    """
    Devuelve True si el texto es None o está vacío.
    """
    return x is None or str(x).strip() == ""


def save_json(obj, path):
    """
    Guarda un objeto Python como JSON.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def load_json(path):
    """
    Carga un archivo JSON.
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_column_names(data):
    """
    Devuelve las columnas tanto si data es:
    - Dataset de Hugging Face
    - lista de diccionarios
    """

    if hasattr(data, "column_names"):
        return list(data.column_names)

    if isinstance(data, list):
        if len(data) == 0:
            return []

        if not isinstance(data[0], dict):
            raise TypeError(
                "Si data es una lista, se esperaba una lista de diccionarios."
            )

        return list(data[0].keys())

    raise TypeError(
        f"Tipo de data no soportado: {type(data)}. "
        "Se esperaba un Dataset de Hugging Face o una lista de diccionarios."
    )


def validate_columns(data, expected_columns):
    """
    Verifica que existan las columnas esperadas.
    Funciona para Dataset de Hugging Face y listas de diccionarios.
    """

    actual_columns = get_column_names(data)

    for col in expected_columns:
        if col not in actual_columns:
            raise ValueError(
                f"Falta la columna esperada: {col}. "
                f"Columnas disponibles: {actual_columns}"
            )

    return actual_columns


def get_sample(data, idx):
    """
    Devuelve una muestra por índice.
    Funciona para Dataset de Hugging Face y listas.
    """

    return data[idx]


def get_texts(data, text_col="impression"):
    """
    Devuelve la columna de texto como lista.
    Funciona para Dataset de Hugging Face y listas de diccionarios.
    """

    validate_columns(data, [text_col])

    if hasattr(data, "column_names"):
        return data[text_col]

    if isinstance(data, list):
        return [sample[text_col] for sample in data]

    raise TypeError(
        f"Tipo de data no soportado: {type(data)}."
    )


def get_valid_indices(data, text_col="impression"):
    """
    Devuelve índices donde la columna de texto elegida no esté vacía.
    Funciona para Dataset de Hugging Face y listas de diccionarios.
    """

    texts = get_texts(data, text_col=text_col)

    return [
        i for i, txt in enumerate(texts)
        if not is_empty_text(txt)
    ]

In [ ]:
%%writefile src/data/split_generator.py
import random
from src.data.utils import save_json, get_valid_indices


def compute_split_sizes(
    n_valid,
    train_size=10000,
    val_size=1000,
    test_size=1000,
    selected_size=30,
    auto_shrink=True,
    train_ratio=0.70,
    val_ratio=0.15
):
    """
    Calcula tamaños de split.

    Si hay suficientes datos, usa los tamaños pedidos.
    Si no hay suficientes datos y auto_shrink=True, usa proporciones.
    """

    needed = train_size + val_size + test_size

    if n_valid >= needed:
        return train_size, val_size, test_size, min(selected_size, test_size)

    if not auto_shrink:
        raise ValueError(
            f"No hay suficientes muestras válidas. "
            f"Disponibles: {n_valid}, necesarias: {needed}"
        )

    new_train_size = int(train_ratio * n_valid)
    new_val_size = int(val_ratio * n_valid)
    new_test_size = n_valid - new_train_size - new_val_size
    new_selected_size = min(selected_size, new_test_size)

    if new_train_size <= 0 or new_val_size <= 0 or new_test_size <= 0:
        raise ValueError(
            f"No hay suficientes muestras válidas para generar splits. "
            f"Muestras válidas: {n_valid}"
        )

    return new_train_size, new_val_size, new_test_size, new_selected_size


def generate_splits(
    hf_split,
    text_col="impression",
    train_size=10000,
    val_size=1000,
    test_size=1000,
    selected_size=30,
    seed=42,
    output_dir="data/splits",
    selected_output="data/selected_indices.json",
    auto_shrink=True
):
    """
    Genera splits reproducibles usando índices.

    Funciona tanto con:
    - Dataset de Hugging Face
    - lista de diccionarios creada con streaming/take()

    Archivos generados:
    - data/splits/train_indices.json
    - data/splits/val_indices.json
    - data/splits/test_indices.json
    - data/selected_indices.json
    """

    valid_indices = get_valid_indices(hf_split, text_col=text_col)

    n_valid = len(valid_indices)

    train_size, val_size, test_size, selected_size = compute_split_sizes(
        n_valid=n_valid,
        train_size=train_size,
        val_size=val_size,
        test_size=test_size,
        selected_size=selected_size,
        auto_shrink=auto_shrink
    )

    rng = random.Random(seed)
    rng.shuffle(valid_indices)

    train_indices = valid_indices[:train_size]

    val_start = train_size
    val_end = train_size + val_size
    val_indices = valid_indices[val_start:val_end]

    test_start = val_end
    test_end = val_end + test_size
    test_indices = valid_indices[test_start:test_end]

    selected_indices = test_indices[:selected_size]

    save_json(train_indices, f"{output_dir}/train_indices.json")
    save_json(val_indices, f"{output_dir}/val_indices.json")
    save_json(test_indices, f"{output_dir}/test_indices.json")
    save_json(selected_indices, selected_output)

    return {
        "train": train_indices,
        "val": val_indices,
        "test": test_indices,
        "selected": selected_indices,
        "n_valid": n_valid,
        "used_train_size": train_size,
        "used_val_size": val_size,
        "used_test_size": test_size,
        "used_selected_size": selected_size,
    }

In [ ]:
%%writefile src/data/dataset.py
from torch.utils.data import Dataset
from src.data.utils import validate_columns, get_sample


class MimicCXRDataset(Dataset):
    """
    Dataset de PyTorch para MIMIC-CXR.

    Funciona tanto con:
    - Dataset de Hugging Face
    - lista de diccionarios creada con streaming/take()

    Devuelve un item compatible con BLIP:
    - pixel_values
    - input_ids
    - attention_mask
    - labels
    - idx
    - text
    """

    def __init__(
        self,
        hf_split,
        indices,
        processor,
        text_col="impression",
        max_length=128
    ):
        self.hf_split = hf_split
        self.indices = indices
        self.processor = processor
        self.text_col = text_col
        self.max_length = max_length

        validate_columns(
            hf_split,
            expected_columns=["image", text_col]
        )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        sample = get_sample(self.hf_split, real_idx)

        image = sample["image"].convert("RGB")
        text = sample[self.text_col]

        encoding = self.processor(
            images=image,
            text=text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {}

        for key, value in encoding.items():
            item[key] = value.squeeze(0)

        labels = item["input_ids"].clone()

        pad_token_id = self.processor.tokenizer.pad_token_id
        labels[labels == pad_token_id] = -100

        item["labels"] = labels
        item["idx"] = real_idx
        item["text"] = text

        return item

In [ ]:
%%writefile src/data/dataloader.py
import torch
from torch.utils.data import DataLoader
from src.data.dataset import MimicCXRDataset
from src.data.utils import load_json


def create_dataloader(
    hf_split,
    indices_path,
    processor,
    text_col="impression",
    batch_size=4,
    shuffle=True,
    num_workers=0,
    max_length=128
):
    """
    Crea un DataLoader para BLIP usando índices guardados en JSON.
    """

    indices = load_json(indices_path)

    dataset = MimicCXRDataset(
        hf_split=hf_split,
        indices=indices,
        processor=processor,
        text_col=text_col,
        max_length=max_length
    )

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    return dataloader

In [ ]:
%%writefile src/visualization/plots.py
import matplotlib.pyplot as plt


def show_sample(sample, title="MIMIC-CXR sample"):
    """
    Muestra una imagen del dataset junto con findings e impression.
    """

    image = sample["image"]
    findings = sample.get("findings")
    impression = sample.get("impression")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image, cmap="gray")
    ax.axis("off")
    ax.set_title(title)
    plt.show()

    print("FINDINGS:")
    print(findings)

    print("\nIMPRESSION:")
    print(impression)


def plot_text_lengths(texts, title="Distribución de longitudes"):
    """
    Grafica distribución de longitudes de textos.
    """

    lengths = [
        len(str(t))
        for t in texts
        if t is not None and str(t).strip() != ""
    ]

    plt.figure(figsize=(8, 4))
    plt.hist(lengths, bins=50)
    plt.title(title)
    plt.xlabel("Cantidad de caracteres")
    plt.ylabel("Frecuencia")
    plt.show()

    print("Cantidad de textos válidos:", len(lengths))
    print("Longitud mínima:", min(lengths))
    print("Longitud máxima:", max(lengths))
    print("Longitud promedio:", sum(lengths) / len(lengths))

In [ ]:
import importlib

import src.data.utils
import src.data.split_generator
import src.data.dataset
import src.data.dataloader

importlib.reload(src.data.utils)
importlib.reload(src.data.split_generator)
importlib.reload(src.data.dataset)
importlib.reload(src.data.dataloader)

print("Módulos recargados correctamente.")

In [ ]:
from src.data.utils import load_mimic_dataset

ds = load_mimic_dataset(cache_dir="data/hf_cache")
train = ds["train"]

print(ds)
print("Columnas:", train.column_names)
print("Features:", train.features)
print("Cantidad de muestras:", len(train))

In [ ]:
#### Esto es para bajar todos los datos

# from src.data.utils import load_mimic_dataset

# ds = load_mimic_dataset(cache_dir="data/hf_cache")
# train = ds["train"]

# print(ds)
# print("Columnas:", train.column_names)
# print("Features:", train.features)
# print("Cantidad de muestras:", len(train))

In [ ]:
from datasets import load_dataset

N_SAMPLES = 200

stream_ds = load_dataset(
    "itsanmolgupta/mimic-cxr-dataset",
    split="train",
    streaming=True
)

train = list(stream_ds.take(N_SAMPLES))

print("Cantidad cargada:", len(train))
print("Columnas:", train[0].keys())
print("Primer impression:")
print(train[0]["impression"])

In [ ]:
expected_columns = ["image", "findings", "impression"]

if isinstance(train, list):
    actual_columns = list(train[0].keys())
else:
    actual_columns = train.column_names

print("Columnas encontradas:", actual_columns)

for col in expected_columns:
    assert col in actual_columns, f"Falta la columna esperada: {col}"

print("Columnas verificadas correctamente.")
print("Importante: la columna correcta es 'impression', en singular.")

In [ ]:
def is_empty_text(x):
    return x is None or str(x).strip() == ""

n = len(train)

missing_findings = sum(is_empty_text(sample["findings"]) for sample in train)
missing_impression = sum(is_empty_text(sample["impression"]) for sample in train)

print("Total de muestras:", n)
print("Findings vacíos:", missing_findings)
print("Impression vacíos:", missing_impression)
print("Findings válidos:", n - missing_findings)
print("Impression válidos:", n - missing_impression)

In [ ]:
import matplotlib.pyplot as plt

findings_lengths = [
    len(str(sample["findings"]))
    for sample in train
    if not is_empty_text(sample["findings"])
]

impression_lengths = [
    len(str(sample["impression"]))
    for sample in train
    if not is_empty_text(sample["impression"])
]

plt.figure(figsize=(8, 4))
plt.hist(findings_lengths, bins=30)
plt.title("Distribución de longitud — findings")
plt.xlabel("Cantidad de caracteres")
plt.ylabel("Frecuencia")
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(impression_lengths, bins=30)
plt.title("Distribución de longitud — impression")
plt.xlabel("Cantidad de caracteres")
plt.ylabel("Frecuencia")
plt.show()

print("Promedio findings:", sum(findings_lengths) / len(findings_lengths))
print("Promedio impression:", sum(impression_lengths) / len(impression_lengths))

In [ ]:
valid_indices = [
    i for i, sample in enumerate(train)
    if not is_empty_text(sample["impression"])
]

print("Cantidad de índices válidos:", len(valid_indices))
print("Primeros 10 índices válidos:", valid_indices[:10])

In [ ]:
from src.data.split_generator import generate_splits

splits = generate_splits(
    hf_split=train,
    text_col="impression",
    train_size=10000,
    val_size=1000,
    test_size=1000,
    selected_size=30,
    seed=42,
    output_dir="data/splits",
    selected_output="data/selected_indices.json",
    auto_shrink=True
)

print("Splits generados correctamente.")
print("Muestras válidas totales:", splits["n_valid"])
print("Train:", len(splits["train"]))
print("Val:", len(splits["val"]))
print("Test:", len(splits["test"]))
print("Selected:", len(splits["selected"]))

print("\nTamaños usados:")
print("used_train_size:", splits["used_train_size"])
print("used_val_size:", splits["used_val_size"])
print("used_test_size:", splits["used_test_size"])
print("used_selected_size:", splits["used_selected_size"])

if isinstance(train, list):
    print("\nModo smoke test detectado: train es una lista.")
    print("Esto está bien para probar el pipeline con N_SAMPLES.")
else:
    print("\nModo dataset completo detectado: train es un Dataset de Hugging Face.")

In [ ]:
from pathlib import Path
from src.data.utils import load_json

split_paths = {
    "train": "data/splits/train_indices.json",
    "val": "data/splits/val_indices.json",
    "test": "data/splits/test_indices.json",
    "selected": "data/selected_indices.json",
}

loaded_splits = {}

for name, path in split_paths.items():
    path = Path(path)

    print("=" * 80)
    print("Split:", name)
    print("Archivo:", path)
    print("Existe:", path.exists())

    assert path.exists(), f"No existe el archivo: {path}"

    indices = load_json(path)
    loaded_splits[name] = indices

    print("Cantidad de índices:", len(indices))
    print("Primeros 10 índices:", indices[:10])

print("=" * 80)
print("Verificando solapamientos...")

train_set = set(loaded_splits["train"])
val_set = set(loaded_splits["val"])
test_set = set(loaded_splits["test"])
selected_set = set(loaded_splits["selected"])

assert train_set.isdisjoint(val_set), "Hay solapamiento entre train y val."
assert train_set.isdisjoint(test_set), "Hay solapamiento entre train y test."
assert val_set.isdisjoint(test_set), "Hay solapamiento entre val y test."
assert selected_set.issubset(test_set), "selected_indices debe ser subconjunto de test_indices."

print("Train ∩ Val:", len(train_set.intersection(val_set)))
print("Train ∩ Test:", len(train_set.intersection(test_set)))
print("Val ∩ Test:", len(val_set.intersection(test_set)))
print("Selected ⊆ Test:", selected_set.issubset(test_set))

print("=" * 80)
print("Verificando que los índices estén dentro del rango del dataset actual...")

n = len(train)

for split_name, indices in loaded_splits.items():
    for idx in indices:
        assert 0 <= idx < n, f"Índice fuera de rango en {split_name}: {idx}"

print("Todos los índices están dentro del rango correcto.")
print("Verificación completa correcta.")

In [ ]:
import importlib

import src.data.utils
import src.data.dataset
import src.data.dataloader
import src.data.split_generator

importlib.reload(src.data.utils)
importlib.reload(src.data.dataset)
importlib.reload(src.data.dataloader)
importlib.reload(src.data.split_generator)

from transformers import BlipProcessor

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

print("Módulos recargados correctamente.")
print("Processor cargado correctamente.")
print(type(processor))

In [ ]:
from src.data.dataloader import create_dataloader

train_loader = create_dataloader(
    hf_split=train,
    indices_path="data/splits/train_indices.json",
    processor=processor,
    text_col="impression",
    batch_size=2,
    shuffle=True,
    num_workers=0,
    max_length=128
)

print("DataLoader creado correctamente.")
print("Cantidad de batches:", len(train_loader))

if isinstance(train, list):
    print("Modo actual: train es una lista de muestras.")
else:
    print("Modo actual: train es un Dataset de Hugging Face.")

print("Cantidad de muestras cargadas en train:", len(train))

In [ ]:
batch = next(iter(train_loader))

print("Batch obtenido correctamente.")
print("Claves del batch:")
print(batch.keys())

print("\nShapes / tipos:")
for key, value in batch.items():
    if hasattr(value, "shape"):
        print(f"{key}: {value.shape}")
    else:
        print(f"{key}: {type(value)} | largo: {len(value)}")

print("\nTextos del batch:")
for i, text in enumerate(batch["text"]):
    print("=" * 80)
    print(f"Ejemplo {i}")
    print(text)

In [ ]:
from src.data.utils import get_sample

print("Inspección detallada del batch")
print("=" * 80)

print("Claves:")
print(batch.keys())

print("\nShapes / tipos:")
for key, value in batch.items():
    if hasattr(value, "shape"):
        print(f"{key}: {value.shape}")
    else:
        print(f"{key}: {type(value)} | largo: {len(value)}")

print("\nÍndices reales del dataset:")
print(batch["idx"])

print("\nTextos originales del batch:")
for i, text in enumerate(batch["text"]):
    print("=" * 80)
    print(f"Ejemplo {i}")
    print(text)

print("\nTokens decodificados desde input_ids:")
for i in range(batch["input_ids"].shape[0]):
    decoded = processor.tokenizer.decode(
        batch["input_ids"][i],
        skip_special_tokens=True
    )

    print("=" * 80)
    print(f"Ejemplo {i}")
    print(decoded)

In [ ]:
import torch
from transformers import BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

model.train()

batch_model = {
    "pixel_values": batch["pixel_values"].to(device),
    "input_ids": batch["input_ids"].to(device),
    "attention_mask": batch["attention_mask"].to(device),
    "labels": batch["labels"].to(device),
}

outputs = model(**batch_model)

print("Forward pass correcto.")
print("Loss:", outputs.loss.item())

In [ ]:
import torch
from src.data.utils import load_json, get_sample

device = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

selected_indices = load_json("data/selected_indices.json")

selected_idx = selected_indices[0]
sample = get_sample(train, selected_idx)

image = sample["image"].convert("RGB")

inputs = processor(
    images=image,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30
    )

generated_caption = processor.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print("Índice seleccionado:", selected_idx)

print("\nCaption generado por BLIP base:")
print(generated_caption)

print("\nImpression real:")
print(sample["impression"])

print("\nFindings real:")
print(sample["findings"])

In [ ]:
import json
from pathlib import Path
from src.data.utils import get_column_names, load_json

output_dir = Path("outputs/dataset_checks")
output_dir.mkdir(parents=True, exist_ok=True)

train_indices = load_json("data/splits/train_indices.json")
val_indices = load_json("data/splits/val_indices.json")
test_indices = load_json("data/splits/test_indices.json")
selected_indices = load_json("data/selected_indices.json")

columns = get_column_names(train)

summary = {
    "dataset": "itsanmolgupta/mimic-cxr-dataset",
    "data_object_type": str(type(train)),
    "n_loaded_samples": len(train),
    "columns": columns,
    "text_col_used": "impression",
    "train_size": len(train_indices),
    "val_size": len(val_indices),
    "test_size": len(test_indices),
    "selected_size": len(selected_indices),
    "batch_keys": list(batch.keys()),
    "batch_shapes": {
        key: list(value.shape)
        for key, value in batch.items()
        if hasattr(value, "shape")
    },
    "forward_pass_loss": float(outputs.loss.item()),
    "example_selected_idx": int(selected_idx),
    "example_generated_caption_blip_base": generated_caption,
    "example_reference_impression": sample["impression"],
    "example_reference_findings": sample["findings"],
}

output_path = output_dir / "dataset_dataloader_verification_summary.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Resumen guardado correctamente en:")
print(output_path)

In [ ]:
print("=" * 80)
print("Conclusión de la exploración del dataset")
print("=" * 80)

print("""
1. El dataset fue cargado correctamente.
2. Se verificó que existen las columnas necesarias: image, findings e impression.
3. La columna usada como texto objetivo en este smoke test fue: impression.
4. Se generaron splits reproducibles:
   - data/splits/train_indices.json
   - data/splits/val_indices.json
   - data/splits/test_indices.json
   - data/selected_indices.json

5. El DataLoader fue creado usando create_dataloader().
6. El Dataset interno MimicCXRDataset procesó correctamente:
   - imagen
   - texto
   - input_ids
   - attention_mask
   - labels

7. El batch generado es compatible con BLIP.
8. Se ejecutó un forward pass con BlipForConditionalGeneration y se obtuvo una loss válida.
9. Se generó un caption de prueba con BLIP base sobre una radiografía seleccionada.
10. El resumen quedó guardado en:
    outputs/dataset_checks/dataset_dataloader_verification_summary.json

Resultado:
La capa de datos queda validada para avanzar al siguiente bloque del proyecto.
""")

if isinstance(train, list):
    print("Modo actual: smoke test con lista de muestras.")
    print("Para entrenamiento real, conviene volver a cargar el dataset completo con load_dataset.")
else:
    print("Modo actual: Dataset completo de Hugging Face.")

print("=" * 80)
print("Próximo paso sugerido: notebook 02_baseline_radiografias.ipynb")
print("Objetivo: correr BLIP base sobre selected_indices.json y guardar captions baseline.")
print("=" * 80)